# Barcelona Attendee Occupancy Stream

Generates changing occupancy signals for CCIB and 14 attendee-relevant hotels, cafes, and interesting spots across Barcelona. Before streaming, each source footprint is converted into a literal GeoJSON silhouette centered on the venue: a stepped hotel with windows, a cafe cup with handle, a wide conference building, or a hollow magnifying glass.

Every five-minute Barcelona-time window applies a synthetic spike or drop to exactly one rotating venue so anomaly detection can be demonstrated. The Eventhouse contract remains `occupancySignalId`, `category`, `currentOccupancy`, `totalOccupancy`, and `geometry`. Capacities, occupancy values, anomalies, and symbol dimensions are demo estimates.

In [1]:
# Install the Azure Event Hubs SDK
%pip install azure-eventhub --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Fabric Eventstream / Azure Event Hub connection ───────────────────────────
# Paste from: Fabric portal -> Eventstream -> Sources -> Custom endpoint
OCCUPANCY_EH_CONN_STR = 'OCCUPANCY_EVENTHUB_CONNECTION_STRING_PLACEHOLDER'
OCCUPANCY_EH_NAME     = 'OCCUPANCY_EVENTHUB_NAME_PLACEHOLDER'

In [3]:
import json
import math
import random
import time
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import pandas as pd

In [4]:


GEOJSON_PATH = Path('/lakehouse/default/Files/occupancy_locations.geojson')
STREAM_INTERVAL_SECONDS = 5
MAX_BATCHES = 120  # 10 minutes at 5-second cadence; use None to run continuously
ANOMALY_WINDOW_MINUTES = 5
ANOMALY_SPIKE_RATIO = 0.96
ANOMALY_DROP_RATIO = 0.04
RANDOM_SEED = 42
BARCELONA_TZ = ZoneInfo('Europe/Madrid')
EXPECTED_LOCATION_COUNT = 15
EXPECTED_COLUMNS = ['occupancySignalId', 'category', 'currentOccupancy', 'totalOccupancy', 'geometry']
ALLOWED_CATEGORIES = {'conference_venue', 'hotel', 'cafe', 'interesting_spot'}

random.seed(RANDOM_SEED)

In [5]:
def category_shape(feature):
    source_ring = feature['geometry']['coordinates'][0]
    longitudes = [point[0] for point in source_ring]
    latitudes = [point[1] for point in source_ring]
    center_lng = (min(longitudes) + max(longitudes)) / 2
    center_lat = (min(latitudes) + max(latitudes)) / 2
    dx, dy = 0.00055, 0.00042

    def point(x, y):
        return [round(center_lng + x * dx, 6), round(center_lat + y * dy, 6)]

    category = feature['properties']['category']
    if category == 'hotel':
        outer = [
            point(-0.75, -1.00), point(0.75, -1.00), point(0.75, 0.65),
            point(0.42, 0.65), point(0.42, 1.00), point(-0.42, 1.00),
            point(-0.42, 0.65), point(-0.75, 0.65), point(-0.75, -1.00),
        ]
        holes = [
            [point(-0.45, -0.55), point(-0.15, -0.55), point(-0.15, -0.15), point(-0.45, -0.15), point(-0.45, -0.55)],
            [point(0.15, -0.55), point(0.45, -0.55), point(0.45, -0.15), point(0.15, -0.15), point(0.15, -0.55)],
            [point(-0.45, 0.10), point(-0.15, 0.10), point(-0.15, 0.45), point(-0.45, 0.45), point(-0.45, 0.10)],
            [point(0.15, 0.10), point(0.45, 0.10), point(0.45, 0.45), point(0.15, 0.45), point(0.15, 0.10)],
        ]
        return {'type': 'Polygon', 'coordinates': [outer, *holes]}

    if category == 'cafe':
        outer = [
            point(-0.85, 0.55), point(0.35, 0.55), point(0.35, 0.38),
            point(0.78, 0.38), point(0.95, 0.20), point(0.95, -0.15),
            point(0.78, -0.35), point(0.42, -0.35), point(0.28, -0.75),
            point(0.05, -0.95), point(-0.50, -0.95), point(-0.72, -0.75),
            point(-0.85, 0.55),
        ]
        handle_hole = [
            point(0.42, 0.20), point(0.70, 0.20), point(0.78, 0.10),
            point(0.78, -0.08), point(0.68, -0.18), point(0.42, -0.18),
            point(0.42, 0.20),
        ]
        return {'type': 'Polygon', 'coordinates': [outer, handle_hole]}

    if category == 'conference_venue':
        outer = [
            point(-1.35, -0.90), point(-0.22, -0.90), point(-0.22, -0.35),
            point(0.22, -0.35), point(0.22, -0.90), point(1.35, -0.90),
            point(1.35, 0.35),
            point(0.95, 0.35), point(0.95, 0.62), point(0.45, 0.62),
            point(0.45, 0.88), point(-0.45, 0.88), point(-0.45, 0.62),
            point(-0.95, 0.62), point(-0.95, 0.35), point(-1.35, 0.35),
            point(-1.35, -0.90),
        ]
        return {'type': 'Polygon', 'coordinates': [outer]}

    outer = [
        point(-0.82, 0.28), point(-0.62, 0.72), point(-0.22, 0.98),
        point(0.28, 0.98), point(0.70, 0.70), point(0.88, 0.28),
        point(0.78, -0.18), point(0.50, -0.48), point(1.10, -0.98),
        point(0.78, -1.25), point(0.18, -0.66), point(-0.25, -0.62),
        point(-0.65, -0.36), point(-0.82, 0.28),
    ]
    lens_hole = [
        point(-0.45, 0.25), point(-0.30, 0.58), point(0.00, 0.75),
        point(0.35, 0.62), point(0.55, 0.30), point(0.45, -0.05),
        point(0.15, -0.28), point(-0.20, -0.22), point(-0.43, 0.02),
        point(-0.45, 0.25),
    ]
    return {'type': 'Polygon', 'coordinates': [outer, lens_hole]}


with GEOJSON_PATH.open(encoding='utf-8') as geojson_file:
    location_collection = json.load(geojson_file)

locations = location_collection['features']
for feature in locations:
    feature['geometry'] = category_shape(feature)

location_ids = [feature['properties']['occupancySignalId'] for feature in locations]
assert location_collection['type'] == 'FeatureCollection'
assert len(locations) == EXPECTED_LOCATION_COUNT
assert len(set(location_ids)) == EXPECTED_LOCATION_COUNT
assert all(feature['properties']['category'] in ALLOWED_CATEGORIES for feature in locations)
assert all(feature['properties']['totalOccupancy'] > 0 for feature in locations)
assert all(feature['geometry']['type'] == 'Polygon' for feature in locations)
assert all(ring[0] == ring[-1] for feature in locations for ring in feature['geometry']['coordinates'])
assert all(len(feature['geometry']['coordinates'][0]) > 5 for feature in locations)

catalog = pd.DataFrame([
    {**feature['properties'], 'geometry': feature['geometry']}
    for feature in locations
])
print(f'Loaded {len(catalog)} occupancy locations with category-shaped polygons')
display(catalog[['occupancySignalId', 'name', 'category', 'totalOccupancy']])

Loaded 15 occupancy locations with category-shaped polygons


In [6]:
def target_occupancy_ratio(category, local_time):
    hour = local_time.hour + local_time.minute / 60
    if category == 'conference_venue':
        arrival = 0.82 * math.exp(-0.5 * ((hour - 9.5) / 1.8) ** 2)
        afternoon = 0.68 * math.exp(-0.5 * ((hour - 15.0) / 2.0) ** 2)
        return min(0.94, 0.05 + max(arrival, afternoon))
    if category == 'hotel':
        overnight = 0.84 if hour < 8 or hour >= 19 else 0.58
        return overnight
    if category == 'cafe':
        morning = 0.72 * math.exp(-0.5 * ((hour - 8.3) / 1.2) ** 2)
        break_time = 0.88 * math.exp(-0.5 * ((hour - 11.0) / 0.8) ** 2)
        lunch = 0.80 * math.exp(-0.5 * ((hour - 13.5) / 1.0) ** 2)
        return min(0.96, 0.08 + max(morning, break_time, lunch))
    daytime = 0.62 * math.exp(-0.5 * ((hour - 15.0) / 3.2) ** 2)
    return min(0.82, 0.05 + daytime)

occupancy_state = {}
for feature in locations:
    props = feature['properties']
    initial_ratio = target_occupancy_ratio(props['category'], datetime.now(BARCELONA_TZ))
    occupancy_state[props['occupancySignalId']] = round(props['totalOccupancy'] * initial_ratio)

In [7]:
def active_anomaly(local_time):
    window_seconds = ANOMALY_WINDOW_MINUTES * 60
    window_number = int(local_time.timestamp() // window_seconds)
    feature = locations[window_number % len(locations)]
    anomaly_kind = 'spike' if window_number % 2 == 0 else 'drop'
    anomaly_ratio = ANOMALY_SPIKE_RATIO if anomaly_kind == 'spike' else ANOMALY_DROP_RATIO
    return feature['properties']['occupancySignalId'], anomaly_kind, anomaly_ratio


def build_occupancy_batch(local_time=None):
    local_time = local_time or datetime.now(BARCELONA_TZ)
    anomaly_id, _, anomaly_ratio = active_anomaly(local_time)
    records = []

    for feature in locations:
        props = feature['properties']
        signal_id = props['occupancySignalId']
        capacity = props['totalOccupancy']
        target = capacity * target_occupancy_ratio(props['category'], local_time)
        previous = occupancy_state[signal_id]
        noise = random.gauss(0, max(1, capacity * 0.012))
        current = round(previous + 0.22 * (target - previous) + noise)

        if signal_id == anomaly_id:
            anomaly_noise = random.gauss(0, capacity * 0.008)
            current = round(capacity * anomaly_ratio + anomaly_noise)

        current = max(0, min(capacity, current))
        occupancy_state[signal_id] = current
        records.append({
            'occupancySignalId': signal_id,
            'category': props['category'],
            'currentOccupancy': current,
            'totalOccupancy': capacity,
            'geometry': feature['geometry'],
        })

    return records


sample_time = datetime.now(BARCELONA_TZ)
sample_batch = build_occupancy_batch(sample_time)
sample_df = pd.DataFrame(sample_batch)
sample_anomaly_id, sample_anomaly_kind, _ = active_anomaly(sample_time)
assert len(sample_df) == EXPECTED_LOCATION_COUNT
assert sample_df.columns.tolist() == EXPECTED_COLUMNS
assert (sample_df['currentOccupancy'].between(0, sample_df['totalOccupancy'])).all()
assert sample_df['occupancySignalId'].is_unique
print(f'Active 5-minute anomaly: {sample_anomaly_kind} at {sample_anomaly_id}')
display(sample_df)

Active 5-minute anomaly: drop at hotel-melia-barcelona-sky


## Category shapes and Eventstream

The streamed `geometry` contains the symbol itself: `hotel` is a stepped hotel with window holes, `cafe` is a cup with a hollow handle, `conference_venue` is a wide conference building with an entrance notch, and `interesting_spot` is a magnifying glass with a hollow lens. Render `geometry` as a filled GeoJSON polygon layer; no separate map icon configuration is required.

Every five-minute Barcelona-time window selects exactly one venue for a synthetic anomaly. Even-numbered windows spike near 96% occupancy and odd-numbered windows drop near 4%; small noise keeps the signal realistic. The anomaly is visible through `currentOccupancy`, so the five-column Eventhouse contract remains unchanged.

Install `azure-eventhub`, then provide the Eventstream connection. Use `dynamic` for `geometry`, `long` for both occupancy values, and `string` for the identifier and category.

In [9]:
from azure.eventhub import EventData, EventHubProducerClient

if not OCCUPANCY_EH_CONN_STR or not OCCUPANCY_EH_NAME:
    raise ValueError('Set OCCUPANCY_EVENTHUB_CONNECTION_STRING and OCCUPANCY_EVENTHUB_NAME before streaming.')

producer = EventHubProducerClient.from_connection_string(
    conn_str=OCCUPANCY_EH_CONN_STR,
    eventhub_name=OCCUPANCY_EH_NAME,
)

batch_number = 0

try:
    while MAX_BATCHES is None or batch_number < MAX_BATCHES:
        batch_time = datetime.now(BARCELONA_TZ)
        records = build_occupancy_batch(batch_time)
        anomaly_id, anomaly_kind, _ = active_anomaly(batch_time)
        event_batch = producer.create_batch()

        for record in records:
            event_batch.add(EventData(json.dumps(record, separators=(',', ':'))))

        producer.send_batch(event_batch)

        batch_number += 1

        print(
            f'Sent batch {batch_number}: {len(records)} signals at {batch_time.isoformat()} '
            f'| anomaly={anomaly_kind}:{anomaly_id}'
        )

        time.sleep(STREAM_INTERVAL_SECONDS)

finally:
    producer.close()

Sent batch 1: 15 signals at 2026-09-21T09:57:43.785831+02:00 | anomaly=drop:hotel-melia-barcelona-sky
Sent batch 2: 15 signals at 2026-09-21T09:57:49.828600+02:00 | anomaly=drop:hotel-melia-barcelona-sky
Sent batch 3: 15 signals at 2026-09-21T09:57:54.860835+02:00 | anomaly=drop:hotel-melia-barcelona-sky
Sent batch 4: 15 signals at 2026-09-21T09:57:59.894394+02:00 | anomaly=drop:hotel-melia-barcelona-sky
Sent batch 5: 15 signals at 2026-09-21T09:58:04.927312+02:00 | anomaly=drop:hotel-melia-barcelona-sky
Sent batch 6: 15 signals at 2026-09-21T09:58:09.935978+02:00 | anomaly=drop:hotel-melia-barcelona-sky
Sent batch 7: 15 signals at 2026-09-21T09:58:14.944159+02:00 | anomaly=drop:hotel-melia-barcelona-sky
Sent batch 8: 15 signals at 2026-09-21T09:58:19.952689+02:00 | anomaly=drop:hotel-melia-barcelona-sky
Sent batch 9: 15 signals at 2026-09-21T09:58:24.962122+02:00 | anomaly=drop:hotel-melia-barcelona-sky
Sent batch 10: 15 signals at 2026-09-21T09:58:29.971134+02:00 | anomaly=drop:hotel